# Step 3 — Model Experiments

Baseline + tuned models for the 3 regression targets (`Spoilage_Risk`, `Efficiency_Ratio`, `Quality_Maintenance_Ratio`) and the classification target (`Vehicle_Type`). Training logic lives in `src/train.py`; this notebook runs it end-to-end and inspects quick-look metrics on the validation split.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score

from train import (
    load_processed,
    train_regression_models,
    train_classification_models,
    save_models,
)
from preprocessor import REGRESSION_TARGETS, CLASSIFICATION_TARGET

## Load processed splits

From `data/processed/` (written by Step 2 — `src/preprocessor.py`).

In [2]:
splits = load_processed()
{k: v.shape for k, v in splits.items()}

{'X_train': (37313, 29),
 'X_val': (7996, 29),
 'X_test': (7996, 29),
 'y_train': (37313, 4),
 'y_val': (7996, 4),
 'y_test': (7996, 4)}

## Train regression models

Baseline (`LinearRegression`) + tuned (`LGBMRegressor`) per target.

In [3]:
regression_models, regression_search_info = train_regression_models(
    splits["X_train"], splits["y_train"]
)
list(regression_models.keys())

['Spoilage_Risk', 'Efficiency_Ratio', 'Quality_Maintenance_Ratio']

## Train classification model

Baseline (`LogisticRegression`) + tuned (`XGBClassifier`, label-encoded) for `Vehicle_Type`.

In [4]:
classification_models, classification_search_info = train_classification_models(
    splits["X_train"], splits["y_train"]
)
list(classification_models[CLASSIFICATION_TARGET].keys())

['baseline', 'tuned']

## Hyperparameter search results

"Tuned" = an actual `RandomizedSearchCV` with cross-validation, not just a fancier algorithm with hardcoded defaults. Best params + CV score per target:

In [5]:
search_rows = []
for target, info in {**regression_search_info, **classification_search_info}.items():
    search_rows.append({
        "target": target,
        "scoring": info["scoring"],
        "best_cv_score": info["best_cv_score"],
        "cv_folds": info["cv_folds"],
        "n_iter": info["n_iter"],
    })
pd.DataFrame(search_rows)

,target,scoring,best_cv_score,cv_folds,n_iter
0,Spoilage_Risk,r2,0.859076,3,15
1,Efficiency_Ratio,r2,0.489443,3,15
2,Quality_Maintenance_Ratio,r2,0.408460,3,15
3,Vehicle_Type,f1_macro,0.428396,3,15


## Quick-look validation metrics

Full evaluation (residual plots, feature importance, confusion matrices) belongs in Step 4 (`src/evaluate.py`) — this is just a sanity check that baseline vs. tuned behaves as expected.

In [6]:
rows = []
for target, variants in regression_models.items():
    for variant, model in variants.items():
        preds = model.predict(splits["X_val"])
        rows.append({
            "target": target,
            "variant": variant,
            "MAE": mean_absolute_error(splits["y_val"][target], preds),
            "R2": r2_score(splits["y_val"][target], preds),
        })
pd.DataFrame(rows)

,target,variant,MAE,R2
0,Spoilage_Risk,baseline,9.391824,0.220102
1,Spoilage_Risk,tuned,4.040810,0.860461
2,Efficiency_Ratio,baseline,4.235752,0.497380
3,Efficiency_Ratio,tuned,4.266745,0.492937
4,Quality_Maintenance_Ratio,baseline,4.311384,0.274793
5,Quality_Maintenance_Ratio,tuned,3.930165,0.400623


In [7]:
rows = []
for variant, model in classification_models[CLASSIFICATION_TARGET].items():
    preds = model.predict(splits["X_val"])
    rows.append({
        "target": CLASSIFICATION_TARGET,
        "variant": variant,
        "accuracy": accuracy_score(splits["y_val"][CLASSIFICATION_TARGET], preds),
        "f1_macro": f1_score(splits["y_val"][CLASSIFICATION_TARGET], preds, average="macro"),
    })
pd.DataFrame(rows)

,target,variant,accuracy,f1_macro
0,Vehicle_Type,baseline,0.460605,0.429713
1,Vehicle_Type,tuned,0.454727,0.421567


## Save models

Writes each target/variant model to `models/<target>_<variant>.joblib`.

In [8]:
all_models = {**regression_models, **classification_models}
save_models(all_models)

## Next steps

Step 4: full evaluation in `src/evaluate.py` — residual plots, feature importance, baseline vs. tuned comparison across all 4 targets.